# 01 — Data Setup & Infrastructure

**Purpose**: Foundation notebook for the wine-diversification correlation analysis.  
All downstream notebooks (02–06) depend on the parquet files saved here.

## Sections
1. Environment setup
2. MotherDuck connection & schema discovery
3. Liv-ex index history (CSV) → `livex_indices_clean.parquet`
4. Liv-ex 1000 components (CSV) → `lx1000_components.parquet`
5. Comparison assets from yfinance → `comparison_assets_monthly.parquet`
6. Latent prices (vintage ≥ 1980, monthly) → `latent_prices_monthly.parquet`
7. Data quality assertions

**All database prices are in GBP unless otherwise stated.**

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
            return parent
    raise RuntimeError(f'Could not find repo root. Searched from: {start}')


_repo_root = str(_find_repo_root(Path.cwd()))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f'Repo root: {_repo_root}')

In [ ]:
import os
import warnings

import duckdb
import pandas as pd
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths — use repo root so this works regardless of CWD when notebook opens
# ---------------------------------------------------------------------------
REPO_ROOT = Path(_repo_root)
DATA_DIR  = REPO_ROOT / "projects" / "correlation-diversification" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_CSV          = DATA_DIR / "liv-ex_index_history.csv"
LX1000_CSV         = DATA_DIR / "25-02-11_lx1000_components_2020-2025.csv"

LIVEX_PARQUET      = DATA_DIR / "livex_indices_clean.parquet"
LX1000_PARQUET     = DATA_DIR / "lx1000_components.parquet"
COMPARISON_PARQUET = DATA_DIR / "comparison_assets_monthly.parquet"
LATENT_PARQUET     = DATA_DIR / "latent_prices_monthly.parquet"

print("Data dir:              ", DATA_DIR)
print("motherduck_token set:  ", bool(os.getenv("motherduck_token")))
print("Liv-ex CSV exists:     ", LIVEX_CSV.exists())
print("LX1000 CSV exists:     ", LX1000_CSV.exists())

## 2. MotherDuck Connection & Schema Discovery

Connect to MotherDuck and discover column names for **both** tables before writing any
analysis queries. Column names are never assumed — all queries use detected names.

**Tables:**
- `winefi.mrt.mrt_unified_trades_tbvm` — cleaned unified trade data
- `dev_winefi_raf.ml.ml_latent_prices_historic` — MCMC latent prices (use for returns)

**Performance rules:** Never `SELECT *` without `LIMIT`. Push all aggregation into SQL.

In [ ]:
con = duckdb.connect("md:")
print("Connected to MotherDuck")

In [ ]:
# Schema discovery: winefi.mrt.mrt_unified_trades_tbvm
trades_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'mrt'
      AND table_name    = 'mrt_unified_trades_tbvm'
    ORDER BY ordinal_position
""").df()

print(f"=== winefi.mrt.mrt_unified_trades_tbvm  ({len(trades_schema)} columns) ===")
display(trades_schema)

In [ ]:
# Schema discovery: dev_winefi_raf.ml.ml_latent_prices_historic
latent_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'dev_winefi_raf'
      AND table_schema  = 'ml'
      AND table_name    = 'ml_latent_prices_historic'
    ORDER BY ordinal_position
""").df()

print(f"=== dev_winefi_raf.ml.ml_latent_prices_historic  ({len(latent_schema)} columns) ===")
display(latent_schema)

In [ ]:
# ---------------------------------------------------------------------------
# Dynamic column detection — never hard-code column names
# ---------------------------------------------------------------------------

def first_col(schema_df, *patterns):
    """Return the first column whose lowercase name contains any of the patterns."""
    for _, row in schema_df.iterrows():
        col_lower = row["column_name"].lower()
        for p in patterns:
            if p.lower() in col_lower:
                return row["column_name"]
    return None


# --- Latent prices table ---
LATENT_LWIN11_COL = first_col(latent_schema, "lwin11", "lwin_11")
LATENT_YYYYMM_COL = first_col(latent_schema, "yyyymm", "yearmonth", "year_month", "date", "month")
LATENT_PRICE_COL  = first_col(latent_schema, "price_latent", "latent_price", "price", "value")
LATENT_REGION_COL = first_col(latent_schema, "region")

print("=== Detected columns: dev_winefi_raf.ml.ml_latent_prices_historic ===")
print(f"  lwin11 : {LATENT_LWIN11_COL}")
print(f"  yyyymm : {LATENT_YYYYMM_COL}")
print(f"  price  : {LATENT_PRICE_COL}")
print(f"  region : {LATENT_REGION_COL}")

if LATENT_LWIN11_COL is None:
    raise ValueError(f"No lwin11 column found. Available: {latent_schema['column_name'].tolist()}")
if LATENT_YYYYMM_COL is None:
    raise ValueError(f"No date column found. Available: {latent_schema['column_name'].tolist()}")
if LATENT_PRICE_COL is None:
    raise ValueError(f"No price column found. Available: {latent_schema['column_name'].tolist()}")

In [ ]:
# Row-level preview with LIMIT — never SELECT * without LIMIT
print("=== winefi.mrt.mrt_unified_trades_tbvm — first 5 rows ===")
trades_preview = con.execute("""
    SELECT * FROM winefi.mrt.mrt_unified_trades_tbvm LIMIT 5
""").df()
display(trades_preview)

print("\n=== dev_winefi_raf.ml.ml_latent_prices_historic — first 5 rows ===")
latent_preview = con.execute("""
    SELECT * FROM dev_winefi_raf.ml.ml_latent_prices_historic LIMIT 5
""").df()
display(latent_preview)

In [ ]:
# Aggregate summaries — all computation stays in SQL
# lwin11 format: 7-char LWIN7 + 4-char vintage year = 11 chars
# vintage extraction: SUBSTRING(lwin11, 8, 4)

latent_summary = con.execute(f"""
    SELECT
        COUNT(*)                                                                   AS row_count,
        COUNT(DISTINCT "{LATENT_LWIN11_COL}")                                     AS unique_lwin11,
        MIN(CAST("{LATENT_YYYYMM_COL}" AS VARCHAR))                               AS min_yyyymm,
        MAX(CAST("{LATENT_YYYYMM_COL}" AS VARCHAR))                               AS max_yyyymm,
        MIN(CAST(SUBSTRING(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 8, 4) AS INT)) AS min_vintage,
        MAX(CAST(SUBSTRING(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 8, 4) AS INT)) AS max_vintage
    FROM dev_winefi_raf.ml.ml_latent_prices_historic
    WHERE CAST(SUBSTRING(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 8, 4) AS INT) >= 1980
""").df()

print("=== Latent prices summary (vintage >= 1980) ===")
display(latent_summary)

## 3. Liv-ex Index History (CSV)

Load `liv-ex_index_history.csv`, filter to 2000+, resample to calendar month-end,
and save as `livex_indices_clean.parquet`. All numeric index columns are preserved.

In [ ]:
try:
    livex_raw = pd.read_csv(LIVEX_CSV, index_col="date", parse_dates=True)
    if livex_raw.index.dtype == "object":
        raise ValueError("date column not parsed as datetime")
except (ValueError, KeyError):
    livex_raw = pd.read_csv(LIVEX_CSV, index_col=0, parse_dates=True)

print("=== Liv-ex CSV — raw structure ===")
print(f"Shape:      {livex_raw.shape}")
print(f"Index type: {livex_raw.index.dtype}")
print(f"Date range: {livex_raw.index.min()} → {livex_raw.index.max()}")
print("\nColumn dtypes:")
print(livex_raw.dtypes)
livex_raw.head(3)

In [ ]:
# Retain only numeric index columns; drop metadata (created_at, updated_at, load_ts)
numeric_cols = livex_raw.select_dtypes(include="number").columns.tolist()
livex_indices = livex_raw[numeric_cols].copy()
livex_indices.index = pd.to_datetime(livex_indices.index)

# Filter to 2000+ and resample to month-end
livex_indices = livex_indices[livex_indices.index >= "2000-01-01"]
livex_monthly = livex_indices.resample("ME").last()

print("=== Liv-ex — monthly aligned ===")
print(f"Shape:      {livex_monthly.shape}")
print(f"Date range: {livex_monthly.index.min().date()} → {livex_monthly.index.max().date()}")

col_doc = pd.DataFrame({
    "column":   livex_monthly.columns,
    "non_null": livex_monthly.notna().sum().values,
    "min":      livex_monthly.min().values.round(2),
    "max":      livex_monthly.max().values.round(2),
})
display(col_doc)

livex_monthly.to_parquet(LIVEX_PARQUET)
print(f"\nSaved → {LIVEX_PARQUET}  ({LIVEX_PARQUET.stat().st_size / 1024:.1f} KB)")

## 4. Liv-ex 1000 Components (CSV)

Load the constituent wine list with LWIN11, LWIN7, producer, vintage, and region metadata.  
Apply **vintage ≥ 1980** filter to retain only modern-era wines.

In [ ]:
lx1000_raw = pd.read_csv(LX1000_CSV)

print("=== LX1000 components — raw ===")
print(f"Shape:   {lx1000_raw.shape}")
print(f"Columns: {lx1000_raw.columns.tolist()}")
display(lx1000_raw.head(3))

In [ ]:
# Apply vintage >= 1980 filter
lx1000 = lx1000_raw[lx1000_raw["vintage"].astype(int) >= 1980].copy()
lx1000["vintage"] = lx1000["vintage"].astype(int)
lx1000["lwin7"]   = lx1000["lwin7"].astype(str).str.strip()
lx1000["lwin11"]  = lx1000["lwin11"].astype(str).str.strip()

print(f"After vintage >= 1980 filter: {len(lx1000)} rows (removed {len(lx1000_raw) - len(lx1000)})")
print(f"Unique LWIN7s:  {lx1000['lwin7'].nunique()}")
print(f"Vintage range:  {lx1000['vintage'].min()} → {lx1000['vintage'].max()}")
print(f"Regions: {sorted(lx1000['region'].unique())}")

lx1000.to_parquet(LX1000_PARQUET, index=False)
print(f"\nSaved → {LX1000_PARQUET}  ({LX1000_PARQUET.stat().st_size / 1024:.1f} KB)")
lx1000.head(3)

## 5. Comparison Assets from yfinance

**Total-return** indices — fair comparison with wine which yields no dividends:

| Ticker | Asset | Currency |
|--------|-------|----------|
| `^SP500TR` | S&P 500 Total Return | USD |
| `CUKX.L` | FTSE 100 Total Return ETF | GBP |
| `GC=F` | Gold front-month futures | USD |

All from 2000-01-01, resampled to calendar month-end.  
Note: `sp500tr` and `gold` are USD-denominated — apply FX conversion in downstream notebooks.

In [ ]:
TICKERS = {
    "sp500tr": "^SP500TR",  # S&P 500 Total Return (USD)
    "ftse100": "CUKX.L",   # FTSE 100 Total Return ETF (GBP)
    "gold":    "GC=F",      # Gold front-month futures (USD)
}
START_DATE = "2000-01-01"

frames = {}
for name, ticker in TICKERS.items():
    raw = yf.download(ticker, start=START_DATE, progress=False, auto_adjust=False)["Close"]
    if isinstance(raw, pd.DataFrame):
        raw = raw.squeeze()
    raw.name = name
    frames[name] = raw
    print(
        f"{name:8s} ({ticker:10s}): {len(raw):4d} daily rows, "
        f"{raw.index.min().date()} → {raw.index.max().date()}, "
        f"nulls={raw.isna().sum()}"
    )

comparison_daily = pd.DataFrame(frames)
print(f"\nCombined daily shape: {comparison_daily.shape}")

In [ ]:
# Resample to month-end
comparison_monthly = comparison_daily.resample("ME").last()
comparison_monthly = comparison_monthly[comparison_monthly.index >= "2000-01-01"]

print("=== Comparison assets — monthly ===")
print(f"Shape:      {comparison_monthly.shape}")
print(f"Date range: {comparison_monthly.index.min().date()} → {comparison_monthly.index.max().date()}")
print()
for col in comparison_monthly.columns:
    s = comparison_monthly[col]
    print(f"  {col:8s}: first={s.dropna().iloc[0]:.2f}  last={s.dropna().iloc[-1]:.2f}  nulls={s.isna().sum()}")

comparison_monthly.to_parquet(COMPARISON_PARQUET)
print(f"\nSaved → {COMPARISON_PARQUET}  ({COMPARISON_PARQUET.stat().st_size / 1024:.1f} KB)")
comparison_monthly.tail(3)

## 6. Latent Prices (vintage ≥ 1980, monthly)

Pull `dev_winefi_raf.ml.ml_latent_prices_historic` with vintage ≥ 1980.

- **`lwin11`**: 11-digit wine-vintage key (LWIN7 + 4-digit vintage year)
- **`yyyymm`**: integer year-month (e.g. `200001` = Jan 2000) — already monthly
- **`price_latent`**: MCMC state-space model price (GBP) — use this for all return calculations
- Vintage is extracted as `SUBSTRING(lwin11, 8, 4)` — all aggregation done in SQL

**All prices are in GBP.**

In [ ]:
# Build the vintage filter expression (applied in SQL — no row retrieval without filter)
VINTAGE_EXPR  = f'CAST(SUBSTRING(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 8, 4) AS INTEGER)'
REGION_SELECT = f'"{LATENT_REGION_COL}"' if LATENT_REGION_COL else 'NULL'

latent_query = f"""
    SELECT
        "{LATENT_LWIN11_COL}"   AS lwin11,
        "{LATENT_YYYYMM_COL}"   AS yyyymm,
        "{LATENT_PRICE_COL}"    AS price_latent,
        {REGION_SELECT}          AS region,
        {VINTAGE_EXPR}           AS vintage
    FROM dev_winefi_raf.ml.ml_latent_prices_historic
    WHERE {VINTAGE_EXPR} >= 1980
    ORDER BY "{LATENT_LWIN11_COL}", "{LATENT_YYYYMM_COL}"
"""

print("Latent prices query:")
print(latent_query)

In [ ]:
latent_df = con.execute(latent_query).df()

# Convert yyyymm integer (e.g. 200001) to month-end date
latent_df["date"] = (
    pd.to_datetime(latent_df["yyyymm"].astype(int).astype(str), format="%Y%m")
    + pd.offsets.MonthEnd(0)
)
latent_df = latent_df.drop(columns=["yyyymm"])

print(f"Latent prices: {latent_df.shape}")
print(f"Date range:    {latent_df['date'].min().date()} → {latent_df['date'].max().date()}")
print(f"Unique lwin11: {latent_df['lwin11'].nunique()}")
print(f"Vintage range: {latent_df['vintage'].min()} → {latent_df['vintage'].max()}")
print()
display(latent_df.head(5))

In [ ]:
latent_df.to_parquet(LATENT_PARQUET, index=False)
print(f"Saved → {LATENT_PARQUET}  ({LATENT_PARQUET.stat().st_size / 1024:.1f} KB)")

# Coverage summary: how many wines have data per month
coverage = latent_df.groupby("date")["lwin11"].count()
print(f"\nMonthly coverage (wines with latent price): "
      f"mean={coverage.mean():.0f}, min={coverage.min()}, max={coverage.max()}")

## 7. Data Quality Assertions

All four parquet files must pass these checks before the notebook is considered complete.

In [ ]:
errors = []

# 1. livex_indices_clean.parquet
_livex = pd.read_parquet(LIVEX_PARQUET)
if len(_livex) < 100:
    errors.append(f"livex_indices_clean has only {len(_livex)} rows (need > 100)")
if _livex.index.min().year > 2001:
    errors.append(f"livex_indices_clean starts in {_livex.index.min().year} (need <= 2001)")
if _livex.index.max().year < 2020:
    errors.append(f"livex_indices_clean ends in {_livex.index.max().year} (need >= 2020)")

# 2. lx1000_components.parquet
_lx1000 = pd.read_parquet(LX1000_PARQUET)
if len(_lx1000) == 0:
    errors.append("lx1000_components is empty")
if "vintage" in _lx1000.columns and int(_lx1000["vintage"].min()) < 1980:
    errors.append(f"lx1000_components contains vintages < 1980 (min={_lx1000['vintage'].min()})")
for col in ["lwin11", "lwin7", "vintage", "region"]:
    if col not in _lx1000.columns:
        errors.append(f"lx1000_components missing column '{col}'")

# 3. comparison_assets_monthly.parquet
_comp = pd.read_parquet(COMPARISON_PARQUET)
if len(_comp) < 100:
    errors.append(f"comparison_assets_monthly has only {len(_comp)} rows (need > 100)")
if _comp.index.min().year > 2001:
    errors.append(f"comparison_assets_monthly starts in {_comp.index.min().year} (need <= 2001)")
for col in ["sp500tr", "ftse100", "gold"]:
    if col not in _comp.columns:
        errors.append(f"comparison_assets_monthly missing column '{col}'")

# 4. latent_prices_monthly.parquet
_latent = pd.read_parquet(LATENT_PARQUET)
if len(_latent) == 0:
    errors.append("latent_prices_monthly is empty")
for col in ["lwin11", "price_latent", "date", "vintage"]:
    if col not in _latent.columns:
        errors.append(f"latent_prices_monthly missing column '{col}'")
if "vintage" in _latent.columns and int(_latent["vintage"].min()) < 1980:
    errors.append(f"latent_prices_monthly contains vintages < 1980 (min={_latent['vintage'].min()})")

# Report
if errors:
    for e in errors:
        print(f"FAIL: {e}")
    raise AssertionError("Data quality checks failed — see output above")
else:
    print("All data quality assertions PASSED.")
    print(f"  livex_indices_clean:      {len(_livex)} rows, "
          f"{_livex.index.min().date()} → {_livex.index.max().date()}")
    print(f"  lx1000_components:        {len(_lx1000)} rows, "
          f"vintage {int(_lx1000['vintage'].min())}–{int(_lx1000['vintage'].max())}")
    print(f"  comparison_assets:        {len(_comp)} rows, "
          f"{_comp.index.min().date()} → {_comp.index.max().date()}")
    print(f"  latent_prices_monthly:    {len(_latent)} rows, "
          f"{_latent['lwin11'].nunique()} unique lwin11s, "
          f"vintage {int(_latent['vintage'].min())}–{int(_latent['vintage'].max())}")

## Summary

| Output file | Description | Frequency | Key columns |
|-------------|-------------|-----------|-------------|
| `livex_indices_clean.parquet` | Liv-ex sub-index history (all numeric columns) | Monthly (ME) | All index columns from CSV |
| `lx1000_components.parquet` | Liv-ex 1000 constituents, vintage ≥ 1980 | Static | `lwin11`, `lwin7`, `name`, `vintage`, `region` |
| `comparison_assets_monthly.parquet` | S&P 500 TR, FTSE 100 TR, Gold — total-return | Monthly (ME) | `sp500tr`, `ftse100`, `gold` |
| `latent_prices_monthly.parquet` | MCMC latent prices, vintage ≥ 1980 | Monthly (ME) | `lwin11`, `price_latent`, `date`, `vintage`, `region` |

**MotherDuck schema documented in Section 2:**
- `winefi.mrt.mrt_unified_trades_tbvm`
- `dev_winefi_raf.ml.ml_latent_prices_historic`

**Currency notes:**
- `price_latent` — GBP (same baseline as Liv-ex)
- `sp500tr` and `gold` — USD (apply FX conversion in downstream notebooks)
- `ftse100` (CUKX.L) — GBP